# Modèle de langage et génération de texte

Dans ces travaux pratiques, nous allons apprendre à un réseau récurrent à écrire « à la manière de Voltaire », caractère par caractère : à partir des 30 caractères précédents, le modèle prédit la probabilité de chaque caractère suivant. C'est un modèle de langage, ici à l'échelle des caractères.

1. Un modèle de langage par caractères.
2. Générer du texte, avec une température.
3. L'entraînement, en regardant ce que le modèle écrit.
4. L'effet de la température.
5. LSTM, GRU ou RNN simple ?
6. Deux couches de LSTM.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Ses entraînements y prennent en tout environ 4 minutes sur une RTX 3090 ; le T4 de Colab est environ deux à trois fois plus lent.

## Récupération des données

Les textes de Voltaire du corpus viennent de [gutenberg.org](https://www.gutenberg.org), sans leurs en-têtes, licences et préfaces : notre modèle de langage ne doit pas apprendre à écrire les mentions légales de Gutenberg ou les préfaces des éditeurs. Nous n'utilisons que `voltaire/voltaire_clean.txt`, qui les réunit. La cellule en affiche les 1 000 premiers caractères.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh voltaire
!ls -l voltaire/
!head -c 1000 voltaire/voltaire_clean.txt

## Import des bibliothèques

In [ ]:
import pathlib

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Préparation des données

La cellule suivante prépare les exemples :

- le texte, en minuscules, dans `text` ;
- le vocabulaire : les caractères du texte, triés, dans `index_to_char` (indice → caractère), et `char_to_index` (caractère → indice) ; `encode` transforme une chaîne en tableau d'indices ;
- les exemples : des séquences de 30 caractères (`maxlen`), prises tous les 3 caractères ; la cible de chacune est le caractère qui la suit ;
- `X`, les séquences encodées, de forme `(nombre de séquences, 30)`, et `y`, les indices des caractères suivants.

In [ ]:
text = pathlib.Path("voltaire/voltaire_clean.txt").read_text(encoding="utf8").lower()
index_to_char = sorted(set(text))
char_to_index = {char: index for index, char in enumerate(index_to_char)}
print(f"{len(text)} caractères : {text[:80]}…")
print(f"{len(index_to_char)} caractères différents : {''.join(index_to_char)}")


def encode(characters: str) -> np.ndarray:
  return np.array([char_to_index[char] for char in characters])


maxlen = 30
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i : i + maxlen])
  next_chars.append(text[i + maxlen])

X = np.stack([encode(sentence) for sentence in sentences])
y = encode(next_chars)
print(f"Formes de X et y : {X.shape}, {y.shape}")
print(f"Premier exemple : « {sentences[0]} » → « {next_chars[0]} », {X[0]} → {y[0]}")

## 1. Un modèle de langage par caractères

Écrivez la fonction `build_model(*recurrent_layers)`, qui renvoie un modèle `keras.Sequential` compilé :

- une entrée `keras.Input` : des séquences de caractères de toute longueur, faites d'indices entiers ;
- une couche `layers.Embedding` : un vecteur de 8 nombres pour chaque caractère du vocabulaire ;
- les couches récurrentes `recurrent_layers`, dans l'ordre ;
- une couche `layers.Dense` qui donne une probabilité à chaque caractère du vocabulaire ;
- compilé avec la perte `sparse_categorical_crossentropy` (les cibles sont des indices), l'optimiseur Adam avec écrêtage de la norme du gradient à 1, et la justesse pour métrique.

Les parties suivantes l'appellent avec la ou les couches récurrentes à comparer, d'abord `build_model(layers.LSTM(128))`.

In [ ]:
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Input(shape=(None,), dtype="int32")`
- `layers.Embedding(len(index_to_char), 8)` : pas de `mask_zero` ici, les séquences n'ont pas de *padding*
- `*recurrent_layers` dans une liste insère les couches une à une
- `layers.Dense(len(index_to_char), activation="softmax")`
- `keras.optimizers.Adam(clipnorm=1.0)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=..., dtype=...),
      layers.Embedding(..., ...),
      *recurrent_layers,
      layers.Dense(..., activation=...),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=...),
    loss=...,
    metrics=["accuracy"],
  )
  return model
```

### Solution

In [ ]:
def build_model(*recurrent_layers: layers.Layer) -> keras.Sequential:
  model = keras.Sequential(
    [
      keras.Input(shape=(None,), dtype="int32"),  # séquences de toute longueur
      layers.Embedding(len(index_to_char), 8),
      *recurrent_layers,
      layers.Dense(len(index_to_char), activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=1.0),  # écrêtage du gradient
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model

La cellule suivante construit le modèle du TP : un LSTM de 128 unités.

In [ ]:
model = build_model(layers.LSTM(128))
model.summary()

## 2. Générer du texte, avec une température

Écrivez la fonction `generate(model, seed, n=200, temperature=1.0)`, qui renvoie le texte `seed` suivi de `n` caractères générés par `model`, un à un :

1. encodez les `maxlen` derniers caractères du texte, et donnez-les au modèle sous la forme d'un batch d'une seule séquence ;
2. le modèle renvoie les probabilités $p_i$ du caractère suivant ; appliquez-leur la température $T$, comme dans les slides : $p_i \propto \exp(z_i / T)$, où les logits $z_i$ valent $\log p_i$ à une constante près, soit $p_i^{1/T}$, normalisées pour que leur somme fasse 1 ;
3. tirez le caractère suivant selon ces probabilités, et ajoutez-le au texte.

La cellule qui suit la solution l'essaie sur le modèle pas encore entraîné.

**Documentation utile**

- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)

In [ ]:
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels à utiliser :

- `generated[-maxlen:]` : les `maxlen` derniers caractères ; `encode(...)[None, :]` ajoute la dimension du batch, de taille 1
- `model(x, training=False)` : un appel direct, plus rapide que `model.predict` pour une seule séquence ; il renvoie un tenseur de forme `(1, len(index_to_char))`, que `keras.ops.convert_to_numpy` transforme en tableau NumPy
- convertissez les probabilités en `float64` avant de les élever à la puissance `1 / temperature` : en `float32`, leur somme s'écarte assez de 1 pour que `np.random.choice` refuse le tirage
- `np.random.choice(len(probas), p=probas)` tire un indice selon les probabilités `probas`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  generated = seed
  for _ in range(n):
    x = encode(generated[...])[None, :]
    probas = keras.ops.convert_to_numpy(model(x, training=False))[0]
    probas = probas.astype("float64") ** ...
    probas /= ...
    generated += index_to_char[np.random.choice(len(probas), p=...)]
  return generated
```

### Solution

In [ ]:
def generate(
  model: keras.Model, seed: str, n: int = 200, temperature: float = 1.0
) -> str:
  generated = seed
  for _ in range(n):
    # Les maxlen derniers caractères, en un batch d'une séquence
    x = encode(generated[-maxlen:])[None, :]
    probas = keras.ops.convert_to_numpy(model(x, training=False))[0]
    # p_i ∝ exp(z_i / T), avec z_i = log p_i à une constante près
    probas = probas.astype("float64") ** (1 / temperature)
    probas /= probas.sum()
    generated += index_to_char[np.random.choice(len(probas), p=probas)]
  return generated

Avant l'entraînement, le modèle donne à peu près la même probabilité à tous les caractères : il écrit n'importe quoi.

In [ ]:
print(generate(model, text[:maxlen], n=100))

## 3. Entraîner le modèle

Le callback `SamplingCallback` fait écrire 200 caractères au modèle toutes les `every` epochs, à partir d'un extrait du texte tiré au hasard : on voit ainsi le modèle progresser.

La fin du corpus, 10 % des séquences (`validation_split=0.1`), sert à la validation : Keras la prend à la fin des données, sans mélanger. Un découpage aléatoire mettrait presque les mêmes séquences des deux côtés, puisque deux séquences voisines ont 27 caractères en commun.

L'entraînement dure 20 epochs : environ une minute sur une RTX 3090, deux à trois sur le T4 de Colab.

In [ ]:
class SamplingCallback(keras.callbacks.Callback):
  def __init__(self, every: int = 5, n: int = 200) -> None:
    super().__init__()
    self.every = every
    self.n = n

  def on_epoch_end(self, epoch: int, logs: dict | None = None) -> None:
    if (epoch + 1) % self.every == 0:
      start = np.random.randint(len(text) - maxlen)
      seed = text[start : start + maxlen]
      print(f"\nAprès {epoch + 1} epochs, à partir de « {seed} » :")
      # self.model : le modèle que fit entraîne
      print(generate(self.model, seed, n=self.n))


history = model.fit(
  X,
  y,
  batch_size=512,
  epochs=20,
  validation_split=0.1,
  callbacks=[SamplingCallback(every=5)],
)

## 4. L'effet de la température

Avec le modèle entraîné, générez 300 caractères à partir du même début de phrase, par exemple `"candide était "`, avec les températures 0,2, 0,5, 1 et 1,5. Comment le texte change-t-il quand la température baisse ? quand elle monte ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Température {temperature} :")
  print(generate(model, "candide était ", n=..., temperature=...))
  print()
```

### Solution

In [ ]:
for temperature in [0.2, 0.5, 1.0, 1.5]:
  print(f"Température {temperature} :")
  print(generate(model, "candide était ", n=300, temperature=temperature))
  print()

À basse température (0,2), le modèle choisit presque toujours le caractère le plus probable : il écrit des mots courts et fréquents, réels ou presque (« monde », « contre », mais aussi « parite »), et tourne vite en boucle sur « de la … de la … ». À haute température (1,5), la distribution s'aplatit : les caractères peu probables sortent plus souvent, et le texte n'est plus qu'une suite de syllabes, de mots inventés et de signes (« & », « ? », « ; »). Entre les deux, vers 0,5, le texte est le plus vraisemblable : des mots plus variés et moins de répétitions.

Aucun de ces textes ne veut rien dire : après 20 epochs, le modèle ne prédit juste le caractère suivant qu'une fois sur deux environ (47 % de justesse de validation). Il a appris les mots courts, des terminaisons françaises (« -ait », « -ent ») et la ponctuation, pas encore les mots longs ni le sens.

## 5. LSTM, GRU ou RNN simple ?

Entraînez trois modèles de 128 unités, un LSTM, un GRU et un RNN simple (`layers.SimpleRNN`), pendant 10 epochs chacun, et comparez leurs pertes de validation (`val_loss`), par exemple sur un même graphique. Quelle cellule fait le mieux ? Le RNN simple est-il en retard, comme la disparition du gradient le laisserait attendre ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `build_model(layers.GRU(128))` : le même modèle, avec un GRU
- `fit` renvoie un objet `History`, dont le dictionnaire `history` donne la perte de validation de chaque epoch, sous la clé `"val_loss"`
- les mêmes paramètres de `fit` que dans la partie 3, sans le callback

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
histories = {}
for name, layer in [("LSTM", layers.LSTM(128)), ("GRU", ...), ("SimpleRNN", ...)]:
  histories[name] = build_model(...).fit(
    X, y, batch_size=512, epochs=10, validation_split=0.1
  )

for name, cell_history in histories.items():
  plt.plot(cell_history.history[...], label=name)
plt.xlabel("Epoch")
plt.ylabel(...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
histories = {}
for name, layer in [
  ("LSTM", layers.LSTM(128)),
  ("GRU", layers.GRU(128)),
  ("SimpleRNN", layers.SimpleRNN(128)),
]:
  histories[name] = build_model(layer).fit(
    X, y, batch_size=512, epochs=10, validation_split=0.1
  )

for name, cell_history in histories.items():
  plt.plot(cell_history.history["val_loss"], label=name)
  print(
    f"{name} : perte de validation minimale {min(cell_history.history['val_loss']):.3f}"
  )
plt.xlabel("Epoch")
plt.ylabel("Perte de validation")
plt.legend()
plt.show()

Après 10 epochs, le GRU est nettement devant (perte de validation minimale d'environ 1,63, mieux que le LSTM de la partie 3 en 20 epochs), et le RNN simple (1,78) devance même le LSTM (1,87). Aucune des trois courbes n'a fini de descendre : ce classement dit surtout quelle cellule apprend le plus vite. Le LSTM a le plus de poids à régler (70 144, contre 52 992 pour le GRU et 17 536 pour le RNN simple), et il démarre lentement.

Le RNN simple n'est pas en retard ici : 30 caractères forment une séquence courte, et la disparition du gradient (il s'atténue d'un pas de temps à l'autre) ne le gêne pas encore beaucoup. Elle le pénalise sur des séquences plus longues, où les portes du LSTM et du GRU laissent passer l'information, et le gradient, sur de plus longues distances. L'écrêtage du gradient (`clipnorm`) le protège de l'autre risque, l'explosion du gradient.

Ces modèles n'ont pas encore surappris : leur perte de validation baisse toujours. Avec beaucoup plus d'epochs, ou un modèle plus gros, elle finirait par remonter alors que celle d'entraînement baisse encore : ce serait le moment d'arrêter l'entraînement (callback [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/)).

## 6. Deux couches de LSTM

Un dernier modèle, donné : deux LSTM empilés. La première couche renvoie un état par caractère (`return_sequences=True`), la séquence que lit la seconde. Comparez sa perte de validation à celle du modèle de la partie 3, et ce qu'il écrit.

In [ ]:
deep_model = build_model(layers.LSTM(128, return_sequences=True), layers.LSTM(128))
deep_model.summary()
deep_history = deep_model.fit(
  X,
  y,
  batch_size=512,
  epochs=20,
  validation_split=0.1,
  callbacks=[SamplingCallback(every=10)],
)
print(
  f"Perte de validation minimale : un LSTM {min(history.history['val_loss']):.3f}, "
  f"deux {min(deep_history.history['val_loss']):.3f}"
)

Les deux couches font mieux que le LSTM seul de la partie 3 : après 20 epochs, une perte de validation minimale d'environ 1,66 contre 1,74. Elles coûtent 2,6 fois plus de paramètres (211 455 contre 79 871) et des epochs plus longues. Le texte, lui, reste fait de mots courts justes et de mots longs inventés : pour écrire des phrases, un modèle de langage par caractères a besoin de bien plus d'entraînement.